In [0]:
import pandas as pd
import numpy as np

In [0]:
def train_prep(train: pd.DataFrame):

    ############################
    # 2️⃣ MERGE STORE INFO
    ############################
    
    store = pd.read_csv("../data/store.csv")
    df = train.merge(store, on="Store", how="left")

    ############################
    # 3️⃣ SORT
    ############################

    df = df.sort_values(["Store", "Date"]).reset_index(drop=True)

    ############################
    # 3️⃣b FILL MISSING DATES
    ############################

    # Build a complete Store × Date grid so that lag/rolling features
    # operate on a contiguous timeline. Missing days are treated as
    # closed (all observation values set to 0).
    min_date = df["Date"].min()
    max_date = df["Date"].max()
    all_dates = pd.date_range(min_date, max_date, freq="D")
    full_index = pd.MultiIndex.from_product(
        [df["Store"].unique(), all_dates], names=["Store", "Date"]
    )
    df = df.set_index(["Store", "Date"]).reindex(full_index).reset_index()

    # Observation columns: 0 for missing days (store was closed / no data)
    obs_cols = ["Sales", "Customers", "Open", "Promo", "SchoolHoliday"]
    df[obs_cols] = df[obs_cols].fillna(0)

    # StateHoliday is a string column at this point; fill with "0" (no holiday)
    df["StateHoliday"] = df["StateHoliday"].fillna("0")

    # Static store columns: carry the store's own value forward (then backward
    # in case a store only appears later in the dataset)
    static_cols = [
        "StoreType", "Assortment",
        "CompetitionDistance", "CompetitionOpenSinceMonth", "CompetitionOpenSinceYear",
        "Promo2", "Promo2SinceWeek", "Promo2SinceYear", "PromoInterval",
    ]
    df[static_cols] = df.groupby("Store")[static_cols].ffill()
    df[static_cols] = df.groupby("Store")[static_cols].bfill()

    # DayOfWeek (Rossmann convention: 1=Mon … 7=Sun) – derive from Date for inserted rows
    df["DayOfWeek"] = (df["Date"].dt.weekday + 1).astype("int8")

    ############################
    # 4️⃣ DATE FEATURES
    ############################

    df["day_of_week"] = (df["Date"].dt.weekday + 1).astype("int8")      # 1–7
    df["day_of_month"] = df["Date"].dt.day.astype("int8")             # 1–31
    df["day_of_year"] = df["Date"].dt.dayofyear.astype("int16")        # 1–365
    df["month"] = df["Date"].dt.month.astype("int8")
    df["week_of_month"] = (
        (df["Date"].dt.day - 1) // 7 + 1
    ).astype("int8")
    df["week_of_year"] = df["Date"].dt.isocalendar().week.astype("int8")
    df["year"] = df["Date"].dt.year.astype("int16")
    df["quarter"] = df["Date"].dt.quarter.astype("int8")
    df["is_month_start"] = df["Date"].dt.is_month_start.astype("int8")
    df["is_month_end"] = df["Date"].dt.is_month_end.astype("int8")

    ################################
    # CYCLICAL ENCODING
    ################################

    # Day of week (7 days)
    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

    # Day of month (31 days)
    df["dom_sin"] = np.sin(2 * np.pi * df["day_of_month"] / 31)
    df["dom_cos"] = np.cos(2 * np.pi * df["day_of_month"] / 31)

    # Day of year (365 days)
    df["doy_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365)
    df["doy_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365)

    # week of month 1-5
    df["wom_sin"] = np.sin(2 * np.pi * df["week_of_month"] / 5)
    df["wom_cos"] = np.cos(2 * np.pi * df["week_of_month"] / 5)

    # week of year
    df["woy_sin"] = np.sin(2 * np.pi * df["week_of_year"] / 52)
    df["woy_cos"] = np.cos(2 * np.pi * df["week_of_year"] / 52)

    # month of year
    df["moy_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["moy_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    for col in ["dow_sin","dow_cos",
                "dom_sin","dom_cos",
                "wom_sin","wom_cos",
                "woy_sin","woy_cos",
                "moy_sin","moy_cos",
                "doy_sin","doy_cos"]:
        df[col] = df[col].astype("float32")

    ############################
    # 5️⃣ COMPETITION FEATURES
    ############################

    # 1. Fill missing distance with a large value
    df["CompetitionDistance"] = df["CompetitionDistance"].fillna(df["CompetitionDistance"].max() * 3)

    # 2. Fill missing open dates with the current record's date 
    # (effectively saying "competition opened today" -> duration = 0)
    df["CompetitionOpenSinceYear"] = df["CompetitionOpenSinceYear"].fillna(df["year"])
    df["CompetitionOpenSinceMonth"] = df["CompetitionOpenSinceMonth"].fillna(df["month"])
    
    df["CompetitionOpenSince"] = (
        12 * (df["year"] - df["CompetitionOpenSinceYear"]) +
        (df["month"] - df["CompetitionOpenSinceMonth"])
    )

    df["CompetitionOpenSince"] = df["CompetitionOpenSince"].clip(lower=0)
    df["CompetitionOpenSince"] = df["CompetitionOpenSince"].fillna(0)

    ############################
    # 6️⃣ PROMO FEATURES
    ############################

    # Fill NaNs for stores that don't participate in Promo2
    df["Promo2SinceYear"] = df["Promo2SinceYear"].fillna(df["year"])
    df["Promo2SinceWeek"] = df["Promo2SinceWeek"].fillna(df["week_of_year"])
    df["PromoInterval"] = df["PromoInterval"].fillna("0,0,0,0") 

    df["Promo2Since"] = (
        12 * (df["year"] - df["Promo2SinceYear"]) +
        (df["week_of_year"] - df["Promo2SinceWeek"])
    )

    df["Promo2Since"] = df["Promo2Since"].clip(lower=0)
    df["Promo2Since"] = df["Promo2Since"].fillna(0)

    promo_map = {
        "Jan,Apr,Jul,Oct": [1,4,7,10],
        "Feb,May,Aug,Nov": [2,5,8,11],
        "Mar,Jun,Sept,Dec": [3,6,9,12]
    }

    df["promo_month_flag"] = 0

    for pattern, months in promo_map.items():
        mask = (
            (df["PromoInterval"] == pattern) &
            (df["Date"].dt.month.isin(months))
        )
        df.loc[mask, "promo_month_flag"] = 1

    df["promo_month_flag"] = df["promo_month_flag"].astype("int8")

    ############################
    # 7️⃣ CATEGORICAL ENCODING
    ############################

    # StateHoliday: 0, a, b, c -> 0, 1, 2, 3
    df["StateHoliday"] = df["StateHoliday"].astype(str).map(
        {"0": 0, "a": 1, "b": 2, "c": 3}
    ).fillna(0).astype("int8")

    # StoreType: a, b, c, d -> 0, 1, 2, 3
    df["StoreType"] = df["StoreType"].map(
        {"a": 0, "b": 1, "c": 2, "d": 3}
    ).astype("int8")

    # Assortment: a, b, c -> 0, 1, 2
    df["Assortment"] = df["Assortment"].map(
        {"a": 0, "b": 1, "c": 2}
    ).astype("int8")
    
    # Drop string columns we don't need anymore
    df = df.drop(columns=["PromoInterval"])

    ############################
    # 8️⃣ LAG FEATURES
    ############################

    lags = [7, 14, 28]

    for lag in lags:
        df[f"lag_{lag}"] = (
            df.groupby("Store")["Sales"]
            .shift(lag)
            .astype("float32")
        )

    # test
    df["lag_customers_7"] = (
        df.groupby("Store")["Customers"]
        .shift(7)
        .astype("float32")
    )

    ############################
    # 9️⃣ ROLLING FEATURES
    ############################

    df["rolling_mean_7"] = (
        df.groupby("Store")["Sales"]
        .shift(7)
        .rolling(7)
        .mean()
        .astype("float32")
    )

    df["rolling_mean_28"] = (
        df.groupby("Store")["Sales"]
        .shift(28)
        .rolling(28)
        .mean()
        .astype("float32")
    )

    ############################
    # 1️⃣1️⃣ DROP EARLY NA ROWS
    ############################

    # Filter by rolling_mean_28 because it has the longest requirement (28 shift + 28 window)
    df = df[df["rolling_mean_28"].notna()].reset_index(drop=True)

    ############################
    # MEMORY CHECK
    ############################

    print("Memory usage (MB):",
        df.memory_usage(deep=True).sum() / 1024**2)

    return df

In [0]:
def test_prep(test: pd.DataFrame):

    ############################
    # 2️⃣ MERGE STORE INFO
    ############################
    
    store = pd.read_csv("../data/store.csv")
    df = test.merge(store, on="Store", how="left")

    ############################
    # 3️⃣ SORT
    ############################

    df = df.sort_values(["Store", "Date"]).reset_index(drop=True)

    ############################
    # 4️⃣ DATE FEATURES
    ############################

    df["day_of_week"] = (df["Date"].dt.weekday + 1).astype("int8")      # 1–7
    df["day_of_month"] = df["Date"].dt.day.astype("int8")             # 1–31
    df["day_of_year"] = df["Date"].dt.dayofyear.astype("int16")        # 1–365
    df["month"] = df["Date"].dt.month.astype("int8")
    df["week_of_month"] = (
        (df["Date"].dt.day - 1) // 7 + 1
    ).astype("int8")
    df["week_of_year"] = df["Date"].dt.isocalendar().week.astype("int8")
    df["year"] = df["Date"].dt.year.astype("int16")
    df["quarter"] = df["Date"].dt.quarter.astype("int8")
    df["is_month_start"] = df["Date"].dt.is_month_start.astype("int8")
    df["is_month_end"] = df["Date"].dt.is_month_end.astype("int8")

    ################################
    # CYCLICAL ENCODING
    ################################

    # Day of week (7 days)
    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

    # Day of month (31 days)
    df["dom_sin"] = np.sin(2 * np.pi * df["day_of_month"] / 31)
    df["dom_cos"] = np.cos(2 * np.pi * df["day_of_month"] / 31)

    # Day of year (365 days)
    df["doy_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365)
    df["doy_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365)

    # week of month 1-5
    df["wom_sin"] = np.sin(2 * np.pi * df["week_of_month"] / 5)
    df["wom_cos"] = np.cos(2 * np.pi * df["week_of_month"] / 5)

    # week of year
    df["woy_sin"] = np.sin(2 * np.pi * df["week_of_year"] / 52)
    df["woy_cos"] = np.cos(2 * np.pi * df["week_of_year"] / 52)

    # month of year
    df["moy_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["moy_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    for col in ["dow_sin","dow_cos",
                "dom_sin","dom_cos",
                "wom_sin","wom_cos",
                "woy_sin","woy_cos",
                "moy_sin","moy_cos",
                "doy_sin","doy_cos"]:
        df[col] = df[col].astype("float32")

    ############################
    # 5️⃣ COMPETITION FEATURES
    ############################

    # 1. Fill missing distance with a large value
    df["CompetitionDistance"] = df["CompetitionDistance"].fillna(df["CompetitionDistance"].max() * 3)

    # 2. Fill missing open dates with the current record's date 
    # (effectively saying "competition opened today" -> duration = 0)
    df["CompetitionOpenSinceYear"] = df["CompetitionOpenSinceYear"].fillna(df["year"])
    df["CompetitionOpenSinceMonth"] = df["CompetitionOpenSinceMonth"].fillna(df["month"])
    
    df["CompetitionOpenSince"] = (
        12 * (df["year"] - df["CompetitionOpenSinceYear"]) +
        (df["month"] - df["CompetitionOpenSinceMonth"])
    )

    df["CompetitionOpenSince"] = df["CompetitionOpenSince"].clip(lower=0)
    df["CompetitionOpenSince"] = df["CompetitionOpenSince"].fillna(0)

    ############################
    # 6️⃣ PROMO FEATURES
    ############################

    # Fill NaNs for stores that don't participate in Promo2
    df["Promo2SinceYear"] = df["Promo2SinceYear"].fillna(df["year"])
    df["Promo2SinceWeek"] = df["Promo2SinceWeek"].fillna(df["week_of_year"])
    df["PromoInterval"] = df["PromoInterval"].fillna("0,0,0,0") 

    df["Promo2Since"] = (
        12 * (df["year"] - df["Promo2SinceYear"]) +
        (df["week_of_year"] - df["Promo2SinceWeek"])
    )

    df["Promo2Since"] = df["Promo2Since"].clip(lower=0)
    df["Promo2Since"] = df["Promo2Since"].fillna(0)

    promo_map = {
        "Jan,Apr,Jul,Oct": [1,4,7,10],
        "Feb,May,Aug,Nov": [2,5,8,11],
        "Mar,Jun,Sept,Dec": [3,6,9,12]
    }

    df["promo_month_flag"] = 0

    for pattern, months in promo_map.items():
        mask = (
            (df["PromoInterval"] == pattern) &
            (df["Date"].dt.month.isin(months))
        )
        df.loc[mask, "promo_month_flag"] = 1

    df["promo_month_flag"] = df["promo_month_flag"].astype("int8")

    ############################
    # 7️⃣ CATEGORICAL ENCODING
    ############################

    # StateHoliday: 0, a, b, c -> 0, 1, 2, 3
    df["StateHoliday"] = df["StateHoliday"].astype(str).map(
        {"0": 0, "a": 1, "b": 2, "c": 3}
    ).fillna(0).astype("int8")

    # StoreType: a, b, c, d -> 0, 1, 2, 3
    df["StoreType"] = df["StoreType"].map(
        {"a": 0, "b": 1, "c": 2, "d": 3}
    ).astype("int8")

    # Assortment: a, b, c -> 0, 1, 2
    df["Assortment"] = df["Assortment"].map(
        {"a": 0, "b": 1, "c": 2}
    ).astype("int8")
    
    # Drop string columns we don't need anymore
    df = df.drop(columns=["PromoInterval"])

    # Missing open is usually 1
    df["Open"] = df["Open"].fillna(1)
    
    ############################
    # MEMORY CHECK
    ############################

    print("Memory usage (MB):",
        df.memory_usage(deep=True).sum() / 1024**2)

    return df

In [0]:
train = pd.read_csv(
    "../data/train.csv",
    parse_dates=["Date"],
    dtype={
        "Store": "int16",
        "DayOfWeek": "int8",
        "Open": "int8",
        "Promo": "int8",
        "SchoolHoliday": "int8"
    },
    low_memory=False
)
#test = pd.read_csv(
#    "../data/test.csv",
#    parse_dates=["Date"],
#    dtype={
#        "Store": "int16",
#        "DayOfWeek": "int8",
#        "Promo": "int8",
#        "SchoolHoliday": "int8"
#    },
#    low_memory=False
#)

train=train_prep(train)
train.index.name="id"
train.to_csv("../data/train_prepared.csv")
#test=test_prep(test)
#test.index.name="id"
#test.to_csv("../data/test_prepared.csv")

# Understandings

## CompetitionDistance == Null

In [0]:
train[train["CompetitionDistance"].isnull()]

In [0]:
train[train["CompetitionDistance"].isnull()]["Store"].unique()

In [0]:
train[train["Store"].isin([291, 622, 879])]

In [0]:
train[["Store","CompetitionDistance"]].drop_duplicates(inplace=False)

## CompetitionOpenSinceMonth == Null

In [0]:
train[train["CompetitionOpenSinceMonth"].isnull()]

In [0]:
train[train["rolling_mean_28"].isnull()]